# LIWO data reconstruction and spatial dependence

I use this notebook to build the research dataset from the LIWO breach-location inventory and scenario table, and then test a basic question: **does flood-consequence similarity have a spatial and flood-defence-system structure that could be useful for generating dependent multi-breach scenarios?**

I start with fairly simple spatial tests before moving toward a graph representation. The aim here is to establish what structure is actually present in the data.

In [ ]:
from pathlib import Path
import json
import requests
import pandas as pd
import numpy as np

# Project paths — keep the notebook portable
ROOT = Path("..").resolve()
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = ROOT / "results"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

WFS_URL = (
    "https://geodata.basisinformatie-overstromingen.nl/geoserver/LIWO_Basis/ows"
)

params = {
    "service": "WFS",
    "version": "2.0.0",
    "request": "GetFeature",
    "typeNames": "LIWO_Basis:gebiedsindeling_doorbraaklocaties_primair",
    "outputFormat": "application/json",
}

response = requests.get(WFS_URL, params=params, timeout=120)
response.raise_for_status()

wfs = response.json()


raw_wfs_path = RAW_DIR / "doorbraaklocaties_primair_wfs.json"
raw_wfs_path.write_text(
    json.dumps(wfs, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("HTTP status:", response.status_code)
print("Server-reported features:", wfs.get("totalFeatures"))
print("Features received:", len(wfs["features"]))

HTTP status: 200
Server-reported features: 633
Features received: 633


## Breach-location table

I first turn the LIWO WFS inventory into a location-level table. I keep the breach identifier, coordinates, river and dike-ring information because these are the variables I will use later to test spatial and system-level relationships.

In [2]:
records = []

for feature in wfs["features"]:
    props = feature.get("properties", {}).copy()
    geometry = feature.get("geometry") or {}

    coords = geometry.get("coordinates", [None, None])

    props["breach_id"] = feature.get("id")
    props["geometry_type"] = geometry.get("type")
    props["geometry_x"] = coords[0] if len(coords) > 0 else None
    props["geometry_y"] = coords[1] if len(coords) > 1 else None

    records.append(props)

df_locations = pd.DataFrame(records)

print("Rows:", len(df_locations))
print("Unique breach IDs:", df_locations["breach_id"].nunique())
print("\nColumns:")
print(df_locations.columns.tolist())

Rows: 633
Unique breach IDs: 633

Columns:
['id', 'breachtypes_id', 'dijkringareas_id', 'openwaters_id', 'name', 'code', 'x', 'y', 'notify', 'lt30', 'f30t300', 'f300t3000', 'f3000t30k', 'gt30k', 'dreigende_overstroming', 'breach_id', 'geometry_type', 'geometry_x', 'geometry_y']


In [3]:
df_locations["breach_id_numeric"] = (
    df_locations["breach_id"]
    .astype(str)
    .str.extract(r"(\d+)$")[0]
    .astype("Int64")
)

print(df_locations[["breach_id", "breach_id_numeric"]].head())

                                        breach_id  breach_id_numeric
0     gebiedsindeling_doorbraaklocaties_primair.1                  1
1     gebiedsindeling_doorbraaklocaties_primair.2                  2
2    gebiedsindeling_doorbraaklocaties_primair.14                 14
3  gebiedsindeling_doorbraaklocaties_primair.1926               1926
4    gebiedsindeling_doorbraaklocaties_primair.25                 25


In [4]:
df_locations = (
    df_locations
    .drop(columns=["breach_id"])
    .rename(columns={"breach_id_numeric": "breach_id"})
)

assert df_locations["breach_id"].notna().all()
assert df_locations["breach_id"].is_unique

print("Final location inventory:", len(df_locations))

Final location inventory: 633


## Merge with the scenario table

I now link the breach-location inventory to the scenario records. This gives me one table in which each scenario has both its flood-scenario information and the spatial/system attributes of the breach location.

In [8]:
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
PROCESSED_DIR = ROOT / "data" / "processed"

df_scenarios = pd.read_csv(
    PROCESSED_DIR / "liwo_scenarios.csv"
)

print("Scenario rows:", len(df_scenarios))
print("Unique scenario IDs:", df_scenarios["scenario_id"].nunique())
print("Unique breach locations:",
      df_scenarios["breach_id"].nunique())

Scenario rows: 1856
Unique scenario IDs: 1856
Unique breach locations: 619


In [9]:
df = df_scenarios.merge(
    df_locations,
    on="breach_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_location")
)

print("Merged rows:", len(df))
print("Unique scenarios:", df["scenario_id"].nunique())
print("Unique breach locations:", df["breach_id"].nunique())

unmatched = df["geometry_x"].isna().sum()
print("Scenarios without matched WFS location:", unmatched)

Merged rows: 1856
Unique scenarios: 1856
Unique breach locations: 619
Scenarios without matched WFS location: 0


In [10]:
print("Columns after merge:")
for col in df.columns:
    print(" -", col)

Columns after merge:
 - breach_id
 - breach_location_id
 - scenario_id
 - scenario_name
 - return_period
 - discharge_m3s
 - breach_name
 - dijkring
 - river
 - damage_MEUR
 - victims
 - affected
 - model_resolution
 - initial_breach_width
 - max_breach_width
 - breach_growth_method
 - scenario_layer
 - map_id
 - id
 - breachtypes_id
 - dijkringareas_id
 - openwaters_id
 - name
 - code
 - x
 - y
 - notify
 - lt30
 - f30t300
 - f300t3000
 - f3000t30k
 - gt30k
 - dreigende_overstroming
 - geometry_type
 - geometry_x
 - geometry_y


In [11]:
print("\nSample:")
display(df.head())


Sample:


,breach_id,breach_location_id,scenario_id,scenario_name,return_period,discharge_m3s,breach_name,dijkring,river,damage_MEUR,...,notify,lt30,f30t300,f300t3000,f3000t30k,gt30k,dreigende_overstroming,geometry_type,geometry_x,geometry_y
0,1,1,15121,20a_Pluimpot 1/400000 Osgetij,400000,NaN,20a_Extrabreslocatie,Dijkring 27 - Tholen en St. Philipsland,Oosterschelde,290,...,None,0,0,0,0,1,0.0,Point,63661,394660
1,14,14,19091,tp-1d Maas 210,125,NaN,Alem,Dijkring 39 - Alem,Maas,52,...,None,0,1,1,1,0,0.0,Point,153030,421657
2,14,14,19092,tp Maas 210,1250,NaN,Alem,Dijkring 39 - Alem,Maas,65,...,None,0,1,1,1,0,0.0,Point,153030,421657
3,14,14,19093,tp+1d Maas 210,12500,NaN,Alem,Dijkring 39 - Alem,Maas,81,...,None,0,1,1,1,0,0.0,Point,153030,421657
4,25,25,5214,Alphen - Min decimeringshoogte,125,NaN,Alphen,Dijkring 41 - Land van Maas en Waal,Maas,1400,...,None,0,1,1,1,0,0.0,Point,157861,425941


## Save the combined research table

I save the merged catalogue so that the rest of the analysis works from one reproducible research table.

In [13]:
catalog_path = PROCESSED_DIR / "liwo_scenario_catalog.csv"

df.to_csv(catalog_path, index=False)

print("Saved")

Saved


## Characterising the scenario catalogue

Before modelling anything, I want to understand how much information the catalogue actually contains. I check the number of scenarios per location, return-period coverage and the distribution of modelled damage. This matters because uneven coverage can affect what kinds of dependence can be estimated reliably.

In [ ]:
# basic scenario coverage
coverage = (
    df.groupby("breach_id")
      .agg(
          n_scenarios=("scenario_id", "nunique"),
          min_return_period=("return_period", "min"),
          max_return_period=("return_period", "max"),
          total_damage_MEUR=("damage_MEUR", "sum"),
      )
      .reset_index()
)

print("Locations with scenarios:", len(coverage))
print("\nScenario-count distribution:")
print(coverage["n_scenarios"].value_counts().sort_index())

print("\nSummary:")
display(coverage.describe())

Locations with scenarios: 619

Scenario-count distribution:
n_scenarios
1     59
2     84
3    297
4    163
5     13
6      1
7      1
8      1
Name: count, dtype: int64

Summary:


,breach_id,n_scenarios,min_return_period,max_return_period
count,619.000000,619.000000,6.190000e+02,6.190000e+02
mean,2056.339257,2.998384,1.813552e+04,2.585766e+05
std,979.049455,0.972933,1.760325e+05,3.222011e+05
min,1.000000,1.000000,-9.999000e+03,-9.999000e+03
25%,1110.500000,3.000000,2.000000e+02,1.250000e+04
50%,2252.000000,3.000000,1.000000e+03,2.000000e+05
75%,2986.500000,4.000000,4.000000e+03,4.000000e+05
max,3475.000000,8.000000,4.000000e+06,4.000000e+06


In [15]:
print("Return-period distribution:")
display(
    df["return_period"]
      .value_counts(dropna=False)
      .sort_index()
      .rename("n_scenarios")
      .to_frame()
)

Return-period distribution:


,n_scenarios
return_period,
-9999,9
50,2
100,9
125,99
200,57
400,142
500,2
1000,34
1250,124


In [ ]:
# make damage explicitly numeric before calculating means
df["damage_MEUR_num"] = pd.to_numeric(
    df["damage_MEUR"],
    errors="coerce"
)

print("Scenarios by river:")

river_summary = (
    df.groupby("river")
      .agg(
          n_scenarios=("scenario_id", "nunique"),
          n_breach_locations=("breach_id", "nunique"),
          mean_damage_MEUR=("damage_MEUR_num", "mean"),
      )
      .sort_values("n_scenarios", ascending=False)
)

display(river_summary)

Scenarios by river:


,n_scenarios,n_breach_locations,mean_damage_MEUR
river,,,
Oosterschelde,226,75,300.951111
Westerschelde,206,61,409.196078
Noordzee,191,57,2033.725275
Waddenzee,164,43,1102.932927
IJssel,106,39,3650.580952
Oude Maas,79,21,1613.936709
Markermeer,70,19,1834.971014
IJsselmeer,63,18,3025.870968
Nieuwe Maas,62,21,3322.400000


In [22]:
print("Damage dtype:", df["damage_MEUR_num"].dtype)
print("Missing damage:", df["damage_MEUR_num"].isna().sum())

display(
    df[
        ["scenario_id", "scenario_name", "damage_MEUR", "damage_MEUR_num"]
    ]
    .sort_values("damage_MEUR_num", ascending=False)
    .head(20)
)

Damage dtype: float64
Missing damage: 26


,scenario_id,scenario_name,damage_MEUR,damage_MEUR_num
29,19054,tp+1d Bemmel,40000,40000.0
547,20674,Amerongen_2000_zand_rep,38000,38000.0
28,21069,tp Bemmel,37000,37000.0
546,20672,Amerongen_1250_zand_rep,36000,36000.0
619,20667,bres15_04A_2000_zand_rep,35000,35000.0
11,19036,tp+1d Angeren,35000,35000.0
908,19055,tp+1d Oosterhout,34000,34000.0
27,19053,tp-1d Bemmel,34000,34000.0
10,21068,tp Angeren,32000,32000.0
1537,20678,Inundatiekanaal_2000_zand_rep,32000,32000.0


In [23]:
print(df["damage_MEUR_num"].describe(percentiles=[
    0.50, 0.75, 0.90, 0.95, 0.99
]))

count     1830.000000
mean      3067.571038
std       5848.025840
min          1.000000
50%        450.000000
75%       2900.000000
90%      10000.000000
95%      16000.000000
99%      28000.000000
max      40000.000000
Name: damage_MEUR_num, dtype: float64


In [24]:
location_cols = [
    "breach_id",
    "geometry_x",
    "geometry_y",
    "breach_name",
    "dijkring",
    "river",
]

available_cols = [c for c in location_cols if c in df.columns]

df_locations_final = (
    df[available_cols]
    .drop_duplicates("breach_id")
    .reset_index(drop=True)
)

print("Unique scenario locations:", len(df_locations_final))
display(df_locations_final.head())

Unique scenario locations: 619


,breach_id,geometry_x,geometry_y,breach_name,dijkring,river
0,1,63661,394660,20a_Extrabreslocatie,Dijkring 27 - Tholen en St. Philipsland,Oosterschelde
1,14,153030,421657,Alem,Dijkring 39 - Alem,Maas
2,25,157861,425941,Alphen,Dijkring 41 - Land van Maas en Waal,Maas
3,185,122028,487923,Amsterdam zuid,Dijkring 44 - Kromme Rijn,Noordzeekanaal
4,191,194414,436878,AngerenARK,"Dijkring 43 - Betuwe, Tieler- en Culemborgerwaard",Pannerdensch kanaal


In [26]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform

coords = df_locations_final[
    ["geometry_x", "geometry_y"]
].astype(float).to_numpy()

distance_matrix = squareform(
    pdist(coords, metric="euclidean")
)

print("Distance matrix shape:", distance_matrix.shape)

nonzero_distances = distance_matrix[distance_matrix > 0]

print(
    "Minimum non-zero distance:",
    nonzero_distances.min(),
    "m"
)

Distance matrix shape: (619, 619)
Minimum non-zero distance: 18.439088914585774 m


In [27]:
k = 5

nearest_neighbors = []

for i, breach_id in enumerate(df_locations_final["breach_id"]):
    distances = distance_matrix[i].copy()
    distances[i] = np.inf

    nearest_idx = np.argsort(distances)[:k]

    for j in nearest_idx:
        nearest_neighbors.append({
            "breach_id": breach_id,
            "neighbor_id": df_locations_final.iloc[j]["breach_id"],
            "distance_m": distances[j],
        })

df_neighbors = pd.DataFrame(nearest_neighbors)

print("Neighbour records:", len(df_neighbors))

display(df_neighbors.head(20))

Neighbour records: 3095


,breach_id,neighbor_id,distance_m
0,1,2368,170.575496
1,1,2367,1175.837574
2,1,2296,3641.577268
3,1,2309,3746.939284
4,1,2310,3956.414159
5,14,974,3207.763863
6,14,1807,3306.927426
7,14,1128,3895.283943
8,14,1719,4901.800588
9,14,1218,5090.395761


In [28]:
location_summary = (
    df.groupby("breach_id")
      .agg(
          n_scenarios=("scenario_id", "nunique"),
          mean_damage=("damage_MEUR_num", "mean"),
          median_damage=("damage_MEUR_num", "median"),
          max_damage=("damage_MEUR_num", "max"),
          mean_affected=("affected", "mean"),
          max_affected=("affected", "max"),
          mean_victims=("victims", "mean"),
          max_victims=("victims", "max"),
      )
      .reset_index()
)

location_summary["log_mean_damage"] = np.log1p(
    location_summary["mean_damage"]
)

print("Location-level rows:", len(location_summary))
display(location_summary.head())

Location-level rows: 619


,breach_id,n_scenarios,mean_damage,median_damage,max_damage,mean_affected,max_affected,mean_victims,max_victims,log_mean_damage
0,1,1,290.000000,290.0,290.0,749.000000,749,4.000000,4,5.673323
1,14,3,66.000000,65.0,81.0,607.666667,632,3.666667,5,4.204693
2,25,3,2366.666667,2200.0,3500.0,24273.333333,34076,244.333333,282,7.769660
3,185,2,17500.000000,17500.0,23000.0,385571.500000,477807,523.000000,729,9.770013
4,191,3,30333.333333,32000.0,35000.0,326231.333333,344660,1199.333333,1466,10.320035


In [29]:
location_features = (
    df_locations_final
    .merge(
        location_summary,
        on="breach_id",
        how="inner",
        validate="one_to_one"
    )
)

print("Location feature rows:", len(location_features))
display(location_features.head())

Location feature rows: 619


,breach_id,geometry_x,geometry_y,breach_name,dijkring,river,n_scenarios,mean_damage,median_damage,max_damage,mean_affected,max_affected,mean_victims,max_victims,log_mean_damage
0,1,63661,394660,20a_Extrabreslocatie,Dijkring 27 - Tholen en St. Philipsland,Oosterschelde,1,290.000000,290.0,290.0,749.000000,749,4.000000,4,5.673323
1,14,153030,421657,Alem,Dijkring 39 - Alem,Maas,3,66.000000,65.0,81.0,607.666667,632,3.666667,5,4.204693
2,25,157861,425941,Alphen,Dijkring 41 - Land van Maas en Waal,Maas,3,2366.666667,2200.0,3500.0,24273.333333,34076,244.333333,282,7.769660
3,185,122028,487923,Amsterdam zuid,Dijkring 44 - Kromme Rijn,Noordzeekanaal,2,17500.000000,17500.0,23000.0,385571.500000,477807,523.000000,729,9.770013
4,191,194414,436878,AngerenARK,"Dijkring 43 - Betuwe, Tieler- en Culemborgerwaard",Pannerdensch kanaal,3,30333.333333,32000.0,35000.0,326231.333333,344660,1199.333333,1466,10.320035


In [30]:
from scipy.spatial.distance import pdist, squareform

feature_map = (
    location_features
    .set_index("breach_id")
)

ids = feature_map.index.to_numpy()

coords = feature_map[
    ["geometry_x", "geometry_y"]
].astype(float).to_numpy()

log_damage = feature_map["log_mean_damage"].to_numpy()

distance_vec = pdist(coords, metric="euclidean")

damage_difference_vec = pdist(
    log_damage.reshape(-1, 1),
    metric="euclidean"
)

spatial_pairs = pd.DataFrame({
    "distance_m": distance_vec,
    "log_damage_difference": damage_difference_vec
})

print("Location pairs:", len(spatial_pairs))
display(spatial_pairs.head())

Location pairs: 191271


,distance_m,log_damage_difference
0,93357.678688,1.468631
1,99257.951626,2.096337
2,110021.333649,4.096690
3,137399.805433,4.646712
4,110191.562839,3.262712


In [31]:
bins = [
    0,
    1_000,
    5_000,
    10_000,
    25_000,
    50_000,
    100_000,
    np.inf
]

labels = [
    "<1 km",
    "1–5 km",
    "5–10 km",
    "10–25 km",
    "25–50 km",
    "50–100 km",
    ">100 km"
]

spatial_pairs["distance_band"] = pd.cut(
    spatial_pairs["distance_m"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

distance_summary = (
    spatial_pairs
    .groupby("distance_band", observed=True)
    .agg(
        n_pairs=("distance_m", "size"),
        median_damage_difference=("log_damage_difference", "median"),
        mean_damage_difference=("log_damage_difference", "mean")
    )
)

display(distance_summary)

,n_pairs,median_damage_difference,mean_damage_difference
distance_band,,,
<1 km,109,1.180335,1.574291
1–5 km,1449,1.305529,1.734698
5–10 km,3375,1.609233,1.981889
10–25 km,17145,1.861550,2.194388
25–50 km,31809,2.035402,2.425470
50–100 km,52056,2.460140,2.824459
>100 km,85328,2.619486,2.903444


In [32]:
# Add river and dike-ring information to every location pair

pair_locations = location_features[
    ["breach_id", "river", "dijkring"]
].copy()

pair_locations = pair_locations.set_index("breach_id")

pairs = spatial_pairs.copy()

# Recover location IDs from the pair construction
n = len(location_features)

pairs["i"] = None
pairs["j"] = None

idx = 0
for i in range(n):
    n_remaining = n - i - 1

    if n_remaining > 0:
        pairs.loc[idx:idx+n_remaining-1, "i"] = i
        pairs.loc[idx:idx+n_remaining-1, "j"] = range(i+1, n)

        idx += n_remaining

pairs["i"] = pairs["i"].astype(int)
pairs["j"] = pairs["j"].astype(int)

location_ids = location_features["breach_id"].to_numpy()

pairs["breach_i"] = location_ids[pairs["i"].to_numpy()]
pairs["breach_j"] = location_ids[pairs["j"].to_numpy()]

pairs["river_i"] = pairs["breach_i"].map(pair_locations["river"])
pairs["river_j"] = pairs["breach_j"].map(pair_locations["river"])

pairs["same_river"] = (
    pairs["river_i"] == pairs["river_j"]
)

pairs["dijkring_i"] = pairs["breach_i"].map(pair_locations["dijkring"])
pairs["dijkring_j"] = pairs["breach_j"].map(pair_locations["dijkring"])

pairs["same_dijkring"] = (
    pairs["dijkring_i"] == pairs["dijkring_j"]
)

print("Total pairs:", len(pairs))
print("Same river:", pairs["same_river"].sum())
print("Same dijkring:", pairs["same_dijkring"].sum())

Total pairs: 191271
Same river: 9908
Same dijkring: 5802


In [33]:
same_river = pairs[pairs["same_river"]].copy()

same_river["distance_band"] = pd.cut(
    same_river["distance_m"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

river_controlled = (
    same_river
    .groupby("distance_band", observed=True)
    .agg(
        n_pairs=("distance_m", "size"),
        median_damage_difference=("log_damage_difference", "median"),
        mean_damage_difference=("log_damage_difference", "mean")
    )
)

display(river_controlled)

,n_pairs,median_damage_difference,mean_damage_difference
distance_band,,,
<1 km,101,1.198649,1.611944
1–5 km,1042,1.276435,1.718016
5–10 km,1582,1.455287,1.839979
10–25 km,3827,1.642657,1.963492
25–50 km,1678,1.725510,2.026711
50–100 km,831,1.881618,2.248399
>100 km,847,1.919045,2.245673


In [34]:
same_dijkring = pairs[pairs["same_dijkring"]].copy()

same_dijkring["distance_band"] = pd.cut(
    same_dijkring["distance_m"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

dijkring_controlled = (
    same_dijkring
    .groupby("distance_band", observed=True)
    .agg(
        n_pairs=("distance_m", "size"),
        median_damage_difference=("log_damage_difference", "median"),
        mean_damage_difference=("log_damage_difference", "mean")
    )
)

display(dijkring_controlled)

,n_pairs,median_damage_difference,mean_damage_difference
distance_band,,,
<1 km,80,0.929017,1.496502
1–5 km,848,1.113650,1.620209
5–10 km,1313,1.320529,1.801771
10–25 km,2226,1.541818,1.839534
25–50 km,974,1.357725,1.667735
50–100 km,262,0.902457,1.113048
>100 km,99,1.429878,1.605862


In [35]:
# Keep only scenarios with a usable return period and damage
analysis = df[
    df["return_period"].notna() &
    df["damage_MEUR_num"].notna()
].copy()

analysis["log_damage"] = np.log1p(
    analysis["damage_MEUR_num"]
)

print("Usable scenario rows:", len(analysis))
print("Return periods:")
display(
    analysis["return_period"]
    .value_counts()
    .sort_index()
)

Usable scenario rows: 1830
Return periods:


return_period
-9999         9
 50           2
 100          9
 125         97
 200         57
 400        136
 500          2
 1000        34
 1250       122
 2000       126
 4000       269
 5000         3
 9000         1
 10000       83
 12499        3
 12500       96
 20000       99
 40000      296
 100000      56
 125000       3
 199999       1
 200000      48
 400000     222
 928315       1
 1000000     51
 4000000      4
Name: count, dtype: int64

In [36]:
rp_coverage = (
    analysis.groupby("return_period")
    .agg(
        n_scenarios=("scenario_id", "nunique"),
        n_locations=("breach_id", "nunique")
    )
    .sort_index()
)

display(rp_coverage)

,n_scenarios,n_locations
return_period,,
-9999,9,9
50,2,2
100,9,9
125,97,97
200,57,53
400,136,135
500,2,2
1000,34,34
1250,122,122


## First spatial question

**Question:** when two breach locations are represented at the same return period, does geographic proximity correspond to more similar damage outcomes?

I start with this simple question because distance is an observable, continuous measure of spatial dependence. If there is no relationship here, a more complicated spatial model would need a different justification. If a relationship exists, I can then test whether river and dike-ring membership explain additional structure.

In [ ]:
# Same-return-period spatial comparison

analysis = df[
    df["return_period"].notna()
    & (df["return_period"] > 0)
    & df["damage_MEUR_num"].notna()
].copy()

analysis["log_damage"] = np.log1p(
    analysis["damage_MEUR_num"]
)

# Return periods with enough observations for a useful analysis
rp_counts = (
    analysis.groupby("return_period")
    .agg(
        n_scenarios=("scenario_id", "nunique"),
        n_locations=("breach_id", "nunique")
    )
    .sort_values("n_locations", ascending=False)
)

print("Return-period coverage after cleaning:")
display(rp_counts)

Return-period coverage after cleaning:


,n_scenarios,n_locations
return_period,,
4000,269,268
40000,296,266
400000,222,221
400,136,135
1250,122,122
2000,126,118
125,97,97
12500,96,96
20000,99,87


In [38]:
usable_rps = (
    rp_counts[
        rp_counts["n_locations"] >= 20
    ]
    .index
    .tolist()
)

print("Usable return periods:")
print(usable_rps)

Usable return periods:
[4000, 40000, 400000, 400, 1250, 2000, 125, 12500, 20000, 10000, 100000, 200, 1000000, 200000, 1000]


In [ ]:
from scipy.spatial.distance import pdist

same_rp_results = []

for rp in usable_rps:

    subset = analysis[
        analysis["return_period"] == rp
    ].copy()

    # One observation per location for this return period and If duplicate rows exist, use the mean.
    subset = (
        subset.groupby("breach_id")
        .agg(
            geometry_x=("geometry_x", "first"),
            geometry_y=("geometry_y", "first"),
            log_damage=("log_damage", "mean"),
            river=("river", "first"),
            dijkring=("dijkring", "first")
        )
        .reset_index()
    )

    if len(subset) < 10:
        continue

    coords = subset[
        ["geometry_x", "geometry_y"]
    ].astype(float).to_numpy()

    damage = subset["log_damage"].to_numpy()

    distances = pdist(coords, metric="euclidean")

    damage_differences = pdist(
        damage.reshape(-1, 1),
        metric="euclidean"
    )

    temp = pd.DataFrame({
        "return_period": rp,
        "distance_m": distances,
        "log_damage_difference": damage_differences
    })

    same_rp_results.append(temp)

same_rp_pairs = pd.concat(
    same_rp_results,
    ignore_index=True
)

print(
    "Same-return-period pairs:",
    len(same_rp_pairs)
)

display(same_rp_pairs.head())

Same-return-period pairs: 140535


,return_period,distance_m,log_damage_difference
0,4000,62164.005357,0.189242
1,4000,238622.005735,4.135833
2,4000,31312.920097,1.791759
3,4000,34834.179149,0.182322
4,4000,210526.602766,1.770706


In [40]:
same_rp_pairs["distance_band"] = pd.cut(
    same_rp_pairs["distance_m"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

same_rp_summary = (
    same_rp_pairs
    .groupby("distance_band", observed=True)
    .agg(
        n_pairs=("distance_m", "size"),
        median_damage_difference=(
            "log_damage_difference",
            "median"
        ),
        mean_damage_difference=(
            "log_damage_difference",
            "mean"
        )
    )
)

display(same_rp_summary)

,n_pairs,median_damage_difference,mean_damage_difference
distance_band,,,
<1 km,206,1.251872,1.722917
1–5 km,3014,1.299283,1.782054
5–10 km,6466,1.556666,1.968827
10–25 km,26707,1.839735,2.159574
25–50 km,39008,1.903351,2.289414
50–100 km,25786,2.169054,2.539441
>100 km,39348,1.972602,2.322622


## Moran's I on scenario severity

I next test spatial autocorrelation at the location level using Moran's I. Here I am asking whether locations with similar average scenario severity tend to be neighbours in space.

In [ ]:
import numpy as np
import pandas as pd

# Build a 5-nearest-neighbour weight matrix


locations = location_features[
    ["breach_id", "geometry_x", "geometry_y", "log_mean_damage"]
].copy().reset_index(drop=True)

coords = locations[
    ["geometry_x", "geometry_y"]
].astype(float).to_numpy()

x = locations["log_mean_damage"].astype(float).to_numpy()

n = len(locations)
k = 5

# Full distances
distance_matrix = squareform(
    pdist(coords, metric="euclidean")
)

# Binary k-nearest-neighbour weights
W = np.zeros((n, n), dtype=float)

for i in range(n):
    d = distance_matrix[i].copy()
    d[i] = np.inf

    neighbors = np.argsort(d)[:k]
    W[i, neighbors] = 1.0

# Row-standardize
row_sums = W.sum(axis=1)
W = W / row_sums[:, None]

print("Locations:", n)
print("Neighbours per location:", k)
print("Total directed links:", int(W.sum()))

Locations: 619
Neighbours per location: 5
Total directed links: 618


In [44]:
print("Total locations:", len(locations))
print("Missing log_mean_damage:", locations["log_mean_damage"].isna().sum())

display(
    locations[
        locations["log_mean_damage"].isna()
    ]
)

Total locations: 619
Missing log_mean_damage: 9


,breach_id,geometry_x,geometry_y,log_mean_damage
47,606,129173,483938,NaN
61,651,64302,409934,NaN
62,652,54835,417069,NaN
63,653,51633,417847,NaN
115,984,56987,416275,NaN
120,989,52540,417398,NaN
192,1251,126662,485857,NaN
278,2163,147191,601713,NaN
288,2179,36928,413995,NaN


In [45]:
missing_ids = locations.loc[
    locations["log_mean_damage"].isna(),
    "breach_id"
].tolist()

print("Locations with missing log damage:", len(missing_ids))

display(
    df[df["breach_id"].isin(missing_ids)][
        [
            "breach_id",
            "scenario_id",
            "scenario_name",
            "return_period",
            "damage_MEUR"
        ]
    ]
)

Locations with missing log damage: 9


,breach_id,scenario_id,scenario_name,return_period,damage_MEUR
133,606,18039,Toetspeil,1250,<1
134,606,18040,Toetspeil+1,12500,<1
187,651,15190,"dkr26_bres1_GrevelingenMeer_22,5km",4000,<1
188,652,15195,dkr26_bres6_GrevelingenMeer_9km,4000,<1
189,653,15196,dkr26_bres7_GrevelingenMeer_7km,4000,<1
327,984,15194,dkr26_bres5_GrevelingenMeer_12km,4000,<1
332,989,15197,dkr26_bres8_GrevelingenMeer_6km,4000,<1
532,1251,18037,Toetspeil,1250,<1
533,1251,18038,Toetspeil+1,12500,<1
783,2163,18034,03_nz_RPp2d_1,200000,<1


In [46]:
locations_moran = locations.dropna(
    subset=["log_mean_damage", "geometry_x", "geometry_y"]
).reset_index(drop=True)

print("Locations used for Moran's I:", len(locations_moran))
print("Locations excluded:", len(locations) - len(locations_moran))

Locations used for Moran's I: 610
Locations excluded: 9


In [47]:
coords_moran = locations_moran[
    ["geometry_x", "geometry_y"]
].astype(float).to_numpy()

x_moran = (
    locations_moran["log_mean_damage"]
    .astype(float)
    .to_numpy()
)

n = len(locations_moran)
k = 5

distance_matrix_moran = squareform(
    pdist(coords_moran, metric="euclidean")
)

W_moran = np.zeros((n, n), dtype=float)

for i in range(n):
    d = distance_matrix_moran[i].copy()
    d[i] = np.inf

    neighbors = np.argsort(d)[:k]
    W_moran[i, neighbors] = 1.0

W_moran = W_moran / W_moran.sum(axis=1)[:, None]

print("New distance matrix:", distance_matrix_moran.shape)
print("Weight matrix:", W_moran.shape)

New distance matrix: (610, 610)
Weight matrix: (610, 610)


In [48]:
observed_I = morans_i(x_moran, W_moran)

print("Observed Moran's I:", observed_I)

Observed Moran's I: 0.5514442647984666


## Permutation significance test

I use a permutation test rather than relying only on the observed Moran's I. I repeatedly shuffle the severity values across locations and recompute Moran's I to see how unusual the observed spatial structure is under a no-spatial-association null model.

In [49]:
rng = np.random.default_rng(42)

n_permutations = 999

permuted_I = np.empty(n_permutations)

for p in range(n_permutations):
    shuffled = rng.permutation(x_moran)
    permuted_I[p] = morans_i(
        shuffled,
        W_moran
    )

# One-sided test for positive spatial autocorrelation
p_value = (
    np.sum(permuted_I >= observed_I) + 1
) / (n_permutations + 1)

print("Observed Moran's I:", observed_I)
print("Permutation p-value:", p_value)

print("\nPermutation null distribution:")
print("Mean:", permuted_I.mean())
print("Std:", permuted_I.std())

print(
    "95% null interval:",
    np.percentile(permuted_I, [2.5, 97.5])
)

Observed Moran's I: 0.5514442647984666
Permutation p-value: 0.001

Permutation null distribution:
Mean: -0.0018086689960857118
Std: 0.024481177364062235
95% null interval: [-0.04911909  0.0486134 ]


In [50]:
z_score = (
    observed_I - permuted_I.mean()
) / permuted_I.std()

print("Permutation z-score:", z_score)

Permutation z-score: 22.59911464089607


## Controlling for river and dike-ring membership

A spatial pattern by itself does not tell me whether geography is the underlying mechanism. Breach locations close to each other are also more likely to share a river or dike ring. I therefore remove the variation explained by river and dike-ring membership and test Moran's I on the residuals.

This is a useful check because it separates broad system-level structure from additional local spatial structure.

In [51]:
import pandas as pd
import numpy as np

# Use the same complete-case locations from the Moran test
loc_model = df_locations_final.merge(
    location_features[["breach_id", "log_mean_damage"]],
    on="breach_id",
    how="inner"
)

loc_model = loc_model.dropna(
    subset=["log_mean_damage", "river", "dijkring", "geometry_x", "geometry_y"]
).reset_index(drop=True)

print("Locations used:", len(loc_model))
print("Rivers:", loc_model["river"].nunique())
print("Dijk rings:", loc_model["dijkring"].nunique())

Locations used: 610
Rivers: 54
Dijk rings: 53


In [52]:
# One-hot encode river and dijkring
X_cat = pd.get_dummies(
    loc_model[["river", "dijkring"]],
    drop_first=True,
    dtype=float
)

X = np.column_stack([
    np.ones(len(loc_model)),
    X_cat.to_numpy()
])

y = loc_model["log_mean_damage"].to_numpy(dtype=float)

# OLS
beta = np.linalg.lstsq(X, y, rcond=None)[0]
y_hat = X @ beta
residuals = y - y_hat

print("Residual mean:", residuals.mean())
print("Residual std:", residuals.std())

Residual mean: 1.956467774673866e-14
Residual std: 1.3134126604613274


In [53]:
coords_resid = loc_model[["geometry_x", "geometry_y"]].to_numpy()

dist_resid = np.sqrt(
    ((coords_resid[:, None, :] - coords_resid[None, :, :]) ** 2).sum(axis=2)
)

k = 5
W_resid = np.zeros_like(dist_resid, dtype=float)

for i in range(len(loc_model)):
    nearest = np.argsort(dist_resid[i])[1:k+1]
    W_resid[i, nearest] = 1.0

# Row-standardize
row_sums = W_resid.sum(axis=1, keepdims=True)
W_resid = np.divide(
    W_resid,
    row_sums,
    out=np.zeros_like(W_resid),
    where=row_sums != 0
)

In [54]:
observed_I_resid = morans_i(residuals, W_resid)

print("Residual Moran's I:", observed_I_resid)

Residual Moran's I: 0.07157298520743394


In [55]:
rng = np.random.default_rng(42)
n_permutations = 999

permuted_I_resid = np.empty(n_permutations)

for p in range(n_permutations):
    shuffled = rng.permutation(residuals)
    permuted_I_resid[p] = morans_i(shuffled, W_resid)

p_value_resid = (
    np.sum(permuted_I_resid >= observed_I_resid) + 1
) / (n_permutations + 1)

z_score_resid = (
    observed_I_resid - permuted_I_resid.mean()
) / permuted_I_resid.std()

print("Residual Moran's I:", observed_I_resid)
print("Permutation p-value:", p_value_resid)
print("Permutation z-score:", z_score_resid)
print("Null mean:", permuted_I_resid.mean())
print("Null std:", permuted_I_resid.std())
print(
    "95% null interval:",
    np.percentile(permuted_I_resid, [2.5, 97.5])
)

Residual Moran's I: 0.07157298520743394
Permutation p-value: 0.004
Permutation z-score: 3.007018307376187
Null mean: -0.0011630366158634537
Null std: 0.024188752574228305
95% null interval: [-0.04393782  0.04815566]


## Interpretation

The first result is that flood-scenario severity is spatially structured. After controlling for river and dike-ring membership, most of the large-scale spatial structure disappears, but a smaller residual dependence remains (Moran's I = 0.072, permutation p = 0.004).

I interpret this as evidence that **system membership explains much of the spatial pattern, while geography still contains some additional information**. That makes it worth testing distance, river and dike-ring relationships jointly instead treating distance as the only source of dependence.

## Next question: does this hold within return periods?

The previous test used location-level severity summaries. I now want to make the comparison more controlled: at the same nominal return period, are nearby breach locations still more similar in consequence severity than distant locations?

This matters because return period itself is a major driver of damage, so I do not want a spatial effect to simply reflect different hazard intensities being mixed together.

In [56]:
rp_counts = (
    df[df["damage_MEUR_num"].notna() & df["return_period"].notna()]
    ["return_period"]
    .value_counts()
    .sort_index()
)

print(rp_counts)

return_period
-9999         9
 50           2
 100          9
 125         97
 200         57
 400        136
 500          2
 1000        34
 1250       122
 2000       126
 4000       269
 5000         3
 9000         1
 10000       83
 12499        3
 12500       96
 20000       99
 40000      296
 100000      56
 125000       3
 199999       1
 200000      48
 400000     222
 928315       1
 1000000     51
 4000000      4
Name: count, dtype: int64


In [57]:
# Keep usable return periods
valid_rps = [
    125, 200, 400, 1000, 1250, 2000, 4000,
    10000, 12500, 20000, 40000, 100000,
    200000, 400000, 1000000
]

rp_df = df[
    df["return_period"].isin(valid_rps) &
    df["damage_MEUR_num"].notna()
].copy()

# One observation per breach location × return period
rp_location = (
    rp_df
    .groupby(["breach_id", "return_period"], as_index=False)
    .agg(
        damage_MEUR=("damage_MEUR_num", "mean"),
        geometry_x=("geometry_x", "first"),
        geometry_y=("geometry_y", "first"),
        river=("river", "first"),
        dijkring=("dijkring", "first")
    )
)

rp_location["log_damage"] = np.log1p(rp_location["damage_MEUR"])

print("Rows:", len(rp_location))
print("Unique locations:", rp_location["breach_id"].nunique())
print("\nObservations by return period:")
print(rp_location["return_period"].value_counts().sort_index())

Rows: 1730
Unique locations: 594

Observations by return period:
return_period
125         97
200         53
400        135
1000        34
1250       122
2000       118
4000       268
10000       79
12500       96
20000       87
40000      266
100000      56
200000      47
400000     221
1000000     51
Name: count, dtype: int64


In [58]:
def moran_permutation_test(values, coords, k=5, n_permutations=999, seed=42):
    values = np.asarray(values, dtype=float)
    coords = np.asarray(coords, dtype=float)

    n = len(values)

    if n <= k:
        return None

    # Pairwise Euclidean distances
    dist = np.sqrt(
        ((coords[:, None, :] - coords[None, :, :]) ** 2).sum(axis=2)
    )

    # K-nearest-neighbour weights
    W = np.zeros_like(dist)

    for i in range(n):
        nearest = np.argsort(dist[i])[1:k+1]
        W[i, nearest] = 1.0

    # Row-standardize
    row_sums = W.sum(axis=1, keepdims=True)
    W = np.divide(
        W,
        row_sums,
        out=np.zeros_like(W),
        where=row_sums != 0
    )

    observed = morans_i(values, W)

    rng = np.random.default_rng(seed)
    permuted = np.empty(n_permutations)

    for p in range(n_permutations):
        permuted[p] = morans_i(
            rng.permutation(values),
            W
        )

    p_value = (
        np.sum(permuted >= observed) + 1
    ) / (n_permutations + 1)

    z_score = (
        observed - permuted.mean()
    ) / permuted.std()

    return {
        "n": n,
        "morans_I": observed,
        "p_value": p_value,
        "z_score": z_score,
        "null_mean": permuted.mean(),
        "null_std": permuted.std(),
        "null_low": np.percentile(permuted, 2.5),
        "null_high": np.percentile(permuted, 97.5)
    }

In [59]:
results = []

for rp in valid_rps:

    sub = rp_location[
        rp_location["return_period"] == rp
    ].dropna(
        subset=["log_damage", "geometry_x", "geometry_y"]
    )

    if len(sub) < 30:
        continue

    result = moran_permutation_test(
        sub["log_damage"].to_numpy(),
        sub[["geometry_x", "geometry_y"]].to_numpy(),
        k=5,
        n_permutations=999,
        seed=42
    )

    result["return_period"] = rp
    results.append(result)

rp_moran_results = (
    pd.DataFrame(results)
    .sort_values("return_period")
    .reset_index(drop=True)
)

rp_moran_results[
    [
        "return_period",
        "n",
        "morans_I",
        "p_value",
        "z_score",
        "null_low",
        "null_high"
    ]
]

,return_period,n,morans_I,p_value,z_score,null_low,null_high
0,125,97,0.324861,0.001,5.529676,-0.114195,0.116568
1,200,53,0.713689,0.001,9.759984,-0.150555,0.140978
2,400,135,0.354476,0.001,7.293312,-0.088003,0.098143
3,1000,34,0.147505,0.043,1.962213,-0.174996,0.179906
4,1250,122,0.523231,0.001,9.878022,-0.106693,0.110333
5,2000,118,0.751409,0.001,14.492454,-0.101104,0.102000
6,4000,268,0.264899,0.001,7.440093,-0.072938,0.068989
7,10000,79,0.354611,0.001,5.662693,-0.126257,0.120206
8,12500,96,0.401414,0.001,6.757023,-0.113966,0.123181
9,20000,87,0.635777,0.001,10.293680,-0.123388,0.132910


## Interpretation

Return-period-stratified analysis gives a more controlled picture. Across the sufficiently populated return periods I tested, Moran's I was positive in every case and statistically significant under permutation testing (p ≤ 0.043), although the strength varied considerably (Moran's I = 0.148–0.751).

I interpret this as evidence that the spatial structure is not coming from one particular return-period class. At the same time, the variation in Moran's I across return periods tells me not to assume a single universal distance effect.

## Next question: can I model the similarity between locations?

The spatial tests tell me that dependence exists, but they do not yet give me a usable mechanism for constructing dependent events. I therefore move from autocorrelation to a pairwise formulation:

**Can I predict whether two breach locations will have similar consequences, conditional on distance and flood-defence-system relationships?**

In [60]:
# Build pairwise same-return-period observations
pair_rows = []

for rp in valid_rps:
    sub = rp_location[
        rp_location["return_period"] == rp
    ][[
        "breach_id",
        "log_damage",
        "geometry_x",
        "geometry_y",
        "river",
        "dijkring"
    ]].dropna().reset_index(drop=True)

    n = len(sub)

    for i in range(n):
        for j in range(i + 1, n):

            dx = sub.loc[i, "geometry_x"] - sub.loc[j, "geometry_x"]
            dy = sub.loc[i, "geometry_y"] - sub.loc[j, "geometry_y"]
            distance_m = np.sqrt(dx**2 + dy**2)

            pair_rows.append({
                "return_period": rp,
                "breach_i": sub.loc[i, "breach_id"],
                "breach_j": sub.loc[j, "breach_id"],
                "distance_km": distance_m / 1000,
                "log_damage_diff": abs(
                    sub.loc[i, "log_damage"] -
                    sub.loc[j, "log_damage"]
                ),
                "same_river": int(
                    sub.loc[i, "river"] == sub.loc[j, "river"]
                ),
                "same_dijkring": int(
                    sub.loc[i, "dijkring"] == sub.loc[j, "dijkring"]
                )
            })

pairs = pd.DataFrame(pair_rows)

print("Pair count:", len(pairs))
pairs.head()

Pair count: 140535


,return_period,breach_i,breach_j,distance_km,log_damage_diff,same_river,same_dijkring
0,125,14,25,6.456874,3.274650,1,0
1,125,14,191,44.094379,6.115559,0,0
2,125,14,205,16.840088,4.566900,1,0
3,125,14,220,29.830679,5.502490,0,0
4,125,14,381,39.265536,6.463853,0,0


In [61]:
distance_bins = [-np.inf, 1, 5, 10, 25, 50, 100, np.inf]
distance_labels = [
    "<1 km",
    "1–5 km",
    "5–10 km",
    "10–25 km",
    "25–50 km",
    "50–100 km",
    ">100 km"
]

pairs["distance_band"] = pd.cut(
    pairs["distance_km"],
    bins=distance_bins,
    labels=distance_labels
)

distance_summary = (
    pairs
    .groupby("distance_band", observed=True)
    .agg(
        n=("log_damage_diff", "size"),
        median_difference=("log_damage_diff", "median"),
        mean_difference=("log_damage_diff", "mean")
    )
    .reset_index()
)

distance_summary

,distance_band,n,median_difference,mean_difference
0,<1 km,206,1.271874,1.727779
1,1–5 km,3014,1.289457,1.774850
2,5–10 km,6466,1.553314,1.969851
3,10–25 km,26707,1.839121,2.160930
4,25–50 km,39008,1.908851,2.294698
5,50–100 km,25786,2.185131,2.553601
6,>100 km,39348,1.976063,2.325516


In [62]:
from scipy.stats import spearmanr

rho, p = spearmanr(
    pairs["distance_km"],
    pairs["log_damage_diff"]
)

print("Spearman rho:", rho)
print("p-value:", p)

Spearman rho: 0.06303878330909263
p-value: 1.041120649009936e-123


In [63]:
pairs["system_same"] = (
    (pairs["same_river"] == 1) |
    (pairs["same_dijkring"] == 1)
).astype(int)

for label, sub in [
    ("All pairs", pairs),
    ("Different rivers", pairs[pairs["same_river"] == 0]),
    ("Different dike rings", pairs[pairs["same_dijkring"] == 0]),
    ("Different river AND dike ring",
     pairs[
         (pairs["same_river"] == 0) &
         (pairs["same_dijkring"] == 0)
     ])
]:

    if len(sub) >= 10:
        rho, p = spearmanr(
            sub["distance_km"],
            sub["log_damage_diff"]
        )

        print(
            f"{label}: "
            f"n={len(sub)}, "
            f"rho={rho:.4f}, "
            f"p={p:.4g}"
        )

All pairs: n=140535, rho=0.0630, p=1.041e-123
Different rivers: n=119719, rho=0.0243, p=3.648e-17
Different dike rings: n=125866, rho=0.0269, p=1.219e-21
Different river AND dike ring: n=112225, rho=0.0113, p=0.0001599


## Interpretation

The pairwise results suggest that a useful dependence model should be **network/system-aware**, instead based on geographic distance alone. I therefore estimate the effect of distance while explicitly controlling for river, dike-ring and return period.

In [64]:
pairs_model = pairs.copy()

# Log distance. +0.1 avoids log(0), although zero-distance
# pairs should normally not exist here.
pairs_model["log_distance"] = np.log1p(pairs_model["distance_km"])

pairs_model["return_period"] = pairs_model["return_period"].astype(str)

print(pairs_model[
    [
        "distance_km",
        "log_distance",
        "log_damage_diff",
        "same_river",
        "same_dijkring",
        "return_period"
    ]
].head())

   distance_km  log_distance  log_damage_diff  same_river  same_dijkring  \
0     6.456874      2.009136         3.274650           1              0   
1    44.094379      3.808758         6.115559           0              0   
2    16.840088      2.881448         4.566900           1              0   
3    29.830679      3.428510         5.502490           0              0   
4    39.265536      3.695496         6.463853           0              0   

  return_period  
0           125  
1           125  
2           125  
3           125  
4           125  


In [65]:
import statsmodels.formula.api as smf

model = smf.ols(
    """
    log_damage_diff
    ~ log_distance
    + same_river
    + same_dijkring
    + C(return_period)
    """,
    data=pairs_model
).fit(
    cov_type="HC3"
)

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:        log_damage_diff   R-squared:                       0.030
Model:                            OLS   Adj. R-squared:                  0.030
Method:                 Least Squares   F-statistic:                     282.3
Date:                Mon, 14 Sep 2026   Prob (F-statistic):               0.00
Time:                        22:48:13   Log-Likelihood:            -2.7507e+05
No. Observations:              140535   AIC:                         5.502e+05
Df Residuals:                  140517   BIC:                         5.504e+05
Df Model:                          17                                         
Covariance Type:                  HC3                                         
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

In [66]:
print(
    "Distance coefficient:",
    model.params["log_distance"]
)

print(
    "Distance p-value:",
    model.pvalues["log_distance"]
)

print(
    "Distance 95% CI:",
    model.conf_int().loc["log_distance"].to_list()
)

print(
    "R-squared:",
    model.rsquared
)

Distance coefficient: 0.015768577876933323
Distance p-value: 0.002265996654370421
Distance 95% CI: [0.005645323074205878, 0.02589183267966077]
R-squared: 0.030005875140616367


In [67]:
model_no_distance = smf.ols(
    """
    log_damage_diff
    ~ same_river
    + same_dijkring
    + C(return_period)
    """,
    data=pairs_model
).fit(
    cov_type="HC3"
)

print("R² without distance:", model_no_distance.rsquared)
print("R² with distance:", model.rsquared)

print(
    "Incremental R² from distance:",
    model.rsquared - model_no_distance.rsquared
)

R² without distance: 0.029944494664279953
R² with distance: 0.030005875140616367
Incremental R² from distance: 6.138047633641452e-05


## Interpretation

The fitted model suggests that flood-consequence similarity is strongly related to hydraulic/system membership, while geographic distance adds a smaller effect after those relationships are controlled for. I therefore treat river and dike-ring relationships as meaningful graph structure instead using distance as the only connection between breach locations.

## Next step: build an interpretable candidate network

I now turn the relationships I have already tested into an explicit network representation:

- geographic proximity;
- same river;
- same dike ring;
- combinations of these relationships.

The goal before training a GNN, first I want to construct a network whose edges have an interpretable basis in the data.

In [68]:
network_features = df_locations_final[
    [
        "breach_id",
        "river",
        "dijkring",
        "geometry_x",
        "geometry_y",
        "breach_name"
    ]
].copy()

print("Locations:", len(network_features))
print("Unique rivers:", network_features["river"].nunique())
print("Unique dike rings:", network_features["dijkring"].nunique())

print("\nTop rivers:")
print(network_features["river"].value_counts().head(15))

print("\nTop dike rings:")
print(network_features["dijkring"].value_counts().head(15))

Locations: 619
Unique rivers: 55
Unique dike rings: 53

Top rivers:
river
Oosterschelde         75
Westerschelde         61
Noordzee              57
Waddenzee             43
IJssel                39
Nieuwe Maas           21
Oude Maas             21
Markermeer            19
Maas                  19
Waal                  18
IJsselmeer            18
Haringvliet           18
Lek                   18
Hollandsche IJssel    15
Boven Rijn            15
Name: count, dtype: int64

Top dike rings:
dijkring
Dijkring 32 - Zeeuwsch Vlaanderen                    41
Dijkring 27 - Tholen en St. Philipsland              33
Dijkring 06 - Friesland en Groningen                 31
Dijkring 13 - Noord-Holland                          30
Dijkring 14 - Zuid-Holland                           28
Dijkring 26 - Schouwen Duivenland                    28
Dijkring 31 - Zuid-Beveland - oost                   24
Dijkring 17 - IJsselmonde                            23
Dijkring 20 - Voorne-Putten                        

In [69]:
locs = network_features.reset_index(drop=True)

coords = locs[["geometry_x", "geometry_y"]].to_numpy()

distance_matrix = np.sqrt(
    ((coords[:, None, :] - coords[None, :, :]) ** 2).sum(axis=2)
)

pair_records = []

for i in range(len(locs)):
    for j in range(i + 1, len(locs)):

        pair_records.append({
            "breach_i": locs.loc[i, "breach_id"],
            "breach_j": locs.loc[j, "breach_id"],
            "distance_km": distance_matrix[i, j] / 1000,
            "same_river": int(
                locs.loc[i, "river"] == locs.loc[j, "river"]
            ),
            "same_dijkring": int(
                locs.loc[i, "dijkring"] == locs.loc[j, "dijkring"]
            )
        })

network_pairs = pd.DataFrame(pair_records)

print("Total location pairs:", len(network_pairs))
print("\nPair types:")
print(
    network_pairs[
        ["same_river", "same_dijkring"]
    ].value_counts()
)

Total location pairs: 191271

Pair types:
same_river  same_dijkring
0           0                178207
1           0                  7262
0           1                  3156
1           1                  2646
Name: count, dtype: int64


In [70]:
network_pairs["distance_band"] = pd.cut(
    network_pairs["distance_km"],
    bins=[0, 1, 5, 10, 25, 50, 100, np.inf],
    labels=[
        "<1 km",
        "1–5 km",
        "5–10 km",
        "10–25 km",
        "25–50 km",
        "50–100 km",
        ">100 km"
    ],
    include_lowest=True
)

network_structure = (
    network_pairs
    .groupby(
        ["distance_band", "same_river", "same_dijkring"],
        observed=True
    )
    .size()
    .reset_index(name="n_pairs")
)

network_structure

,distance_band,same_river,same_dijkring,n_pairs
0,<1 km,0,0,4
1,<1 km,0,1,4
2,<1 km,1,0,25
3,<1 km,1,1,76
4,1–5 km,0,0,206
5,1–5 km,0,1,201
6,1–5 km,1,0,395
7,1–5 km,1,1,647
8,5–10 km,0,0,1176
9,5–10 km,0,1,617


## What the distance bands tell me

The short-distance pairs are heavily concentrated within the same river and dike-ring systems. For example, among the 109 pairs within 1 km, 76 are in the same river and dike ring, 25 share only the river, and 4 share only the dike ring.

At distances above 100 km, the overwhelming majority of pairs share neither system relationship.

This tells me that **distance and system membership are strongly entangled**. That is important for the model design: a distance-only model may partly act as a proxy for system membership, so I should represent both explicitly.

## Turning the relationships into graph edges

At this point I have three interpretable ingredients: distance, same-river membership and same-dike-ring membership. I combine them into a simple edge score so that I can inspect the strongest candidate connections and later provide the GNN with an explicit graph structure.

I use a simple score here deliberately. It is a transparent starting point, not a claim that this is the final or optimal dependence function.

In [71]:
graph_edges = network_pairs.copy()

# Distance-based proximity
graph_edges["distance_weight"] = np.exp(
    -graph_edges["distance_km"] / 25.0
)

# Simple interpretable relationship score
graph_edges["edge_score"] = (
    1.0 * graph_edges["distance_weight"]
    + 2.0 * graph_edges["same_river"]
    + 3.0 * graph_edges["same_dijkring"]
)

print(
    graph_edges[
        [
            "breach_i",
            "breach_j",
            "distance_km",
            "same_river",
            "same_dijkring",
            "edge_score"
        ]
    ]
    .sort_values("edge_score", ascending=False)
    .head(20)
)

        breach_i  breach_j  distance_km  same_river  same_dijkring  edge_score
180245      3011      3012     0.018439           1              1    5.999263
387            1      2368     0.170575           1              1    5.993200
7350         383       384     0.183709           1              1    5.992679
37036        653       988     0.239835           1              1    5.990452
127010      1934      1935     0.253103           1              1    5.989927
135881      2171      3210     0.283679           1              1    5.988717
118300      1810      2550     0.348080           1              1    5.986173
59609        966      2171     0.390881           1              1    5.984486
184131      3180      3181     0.398121           1              1    5.984201
190325      3346      3347     0.416368           1              1    5.983483
125930      1931      1932     0.420219           1              1    5.983332
158118      2341      2342     0.425183           1 

In [72]:
print(
    graph_edges["edge_score"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]
    )
)

count    191271.000000
mean          0.316929
std           0.892695
min           0.000002
50%           0.030539
90%           0.494931
95%           2.419379
99%           5.552858
99.9%         5.927133
max           5.999263
Name: edge_score, dtype: float64


In [74]:
graph_edges["relationship_type"] = np.select(
    [
        (graph_edges["same_river"] == 1) &
        (graph_edges["same_dijkring"] == 1),
        (graph_edges["same_river"] == 1),
        (graph_edges["same_dijkring"] == 1)
    ],
    [
        "same_river_and_dijkring",
        "same_river",
        "same_dijkring"
    ],
    default="spatial_only"
)

print(
    graph_edges["relationship_type"]
    .value_counts()
)

relationship_type
spatial_only               178207
same_river                   7262
same_dijkring                3156
same_river_and_dijkring      2646
Name: count, dtype: int64


## Testing consequence similarity by relationship type

I now compare observed consequence similarity for different relationship types. This is a direct check of whether the graph relationships I am about to encode correspond to smaller differences in scenario severity.

In [76]:
pairs["relationship_type"] = np.select(
    [
        (pairs["same_river"] == 1) &
        (pairs["same_dijkring"] == 1),
        pairs["same_river"] == 1,
        pairs["same_dijkring"] == 1
    ],
    [
        "same_river_and_dijkring",
        "same_river",
        "same_dijkring"
    ],
    default="spatial_only"
)

In [77]:
relationship_similarity = (
    pairs
    .groupby("relationship_type", observed=True)
    .agg(
        n_pairs=("log_damage_diff", "size"),
        median_log_damage_diff=("log_damage_diff", "median"),
        mean_log_damage_diff=("log_damage_diff", "mean"),
        q25=("log_damage_diff", lambda x: x.quantile(0.25)),
        q75=("log_damage_diff", lambda x: x.quantile(0.75))
    )
    .sort_values("median_log_damage_diff")
)

relationship_similarity

,n_pairs,median_log_damage_diff,mean_log_damage_diff,q25,q75
relationship_type,,,,,
same_river_and_dijkring,7175,1.155368,1.542568,0.469931,2.262590
same_dijkring,7494,1.530717,1.939839,0.709107,2.799979
same_river,13641,1.758602,2.052521,0.866811,2.997516
spatial_only,112225,2.036882,2.400659,0.960936,3.488394


In [78]:
local_pairs = pairs[pairs["distance_km"] <= 25].copy()

local_relationship_similarity = (
    local_pairs
    .groupby("relationship_type", observed=True)
    .agg(
        n_pairs=("log_damage_diff", "size"),
        median_log_damage_diff=("log_damage_diff", "median"),
        mean_log_damage_diff=("log_damage_diff", "mean")
    )
    .sort_values("median_log_damage_diff")
)

local_relationship_similarity

,n_pairs,median_log_damage_diff,mean_log_damage_diff
relationship_type,,,
same_river_and_dijkring,6182,1.197519,1.586247
same_dijkring,4938,1.654371,2.094227
same_river,9401,1.776265,2.085959
spatial_only,15872,1.949246,2.293143


In [79]:
rp_relationship = (
    pairs
    .groupby(
        ["return_period", "relationship_type"],
        observed=True
    )
    .agg(
        n_pairs=("log_damage_diff", "size"),
        median_diff=("log_damage_diff", "median")
    )
    .reset_index()
)

rp_relationship.head(30)

,return_period,relationship_type,n_pairs,median_diff
0,125,same_dijkring,178,0.864007
1,125,same_river,411,2.028423
2,125,same_river_and_dijkring,170,0.753590
3,125,spatial_only,3897,2.071527
4,200,same_dijkring,112,1.247896
5,200,same_river,55,0.693097
6,200,same_river_and_dijkring,74,0.772683
7,200,spatial_only,1137,2.845690
8,400,same_dijkring,461,1.333693
9,400,same_river,1395,1.945910


## Interpretation

The return-period-specific results generally show the same qualitative pattern: pairs sharing both a river and a dike ring tend to have smaller consequence differences than pairs connected only by space, although some smaller groups are noisy.

I do not treat any one return period as definitive. The useful result for the next modelling step is that **system relationships contain information about consequence similarity**, which gives a substantive reason to encode them as graph features.

## Graph design

I now have enough evidence to define the first graph representation.

**Node:** one breach location.

**Edge information:**
- spatial proximity;
- same river;
- same dike ring.

**Node information:**
- location;
- scenario severity;
- return period;
- river;
- dike ring.

**Learning target:** consequence similarity / severity relationships between locations.

This is the point where the earlier exploratory analysis becomes a concrete graph-learning problem.

In [ ]:
# Data leakage issue: split by breach location, because the same location appears across return periods.
from sklearn.model_selection import GroupShuffleSplit

model_data = rp_location.dropna(
    subset=[
        "log_damage",
        "geometry_x",
        "geometry_y",
        "river",
        "dijkring"
    ]
).copy()

# Split by breach location so the same location cannot appear in both train and test sets.
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_data,
        groups=model_data["breach_id"]
    )
)

train = model_data.iloc[train_idx].copy()
test = model_data.iloc[test_idx].copy()

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Train locations:", train["breach_id"].nunique())
print("Test locations:", test["breach_id"].nunique())

Train rows: 1382
Test rows: 348
Train locations: 475
Test locations: 119


## First predictive test

Before moving to a GNN, I want a simple benchmark. I test whether distance, river and dike-ring relationships can help predict the log damage of an **unseen breach location**.

The holdout is done by breach location rather than by individual scenario row because the same location appears across multiple return periods. A row-wise random split would therefore allow information from the same physical location to appear in both training and test data.

In [82]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

# Make sure return period is numeric
train["return_period"] = pd.to_numeric(train["return_period"])
test["return_period"] = pd.to_numeric(test["return_period"])

# Features derived only from TRAIN locations
def build_neighbor_features(test_df, train_df, k=5):

    output = []

    train_coords = train_df[
        ["geometry_x", "geometry_y"]
    ].to_numpy()

    for _, row in test_df.iterrows():

        # Same return-period training scenarios
        candidates = train_df[
            train_df["return_period"] == row["return_period"]
        ].copy()

        if len(candidates) == 0:
            candidates = train_df.copy()

        candidate_coords = candidates[
            ["geometry_x", "geometry_y"]
        ].to_numpy()

        dx = candidate_coords[:, 0] - row["geometry_x"]
        dy = candidate_coords[:, 1] - row["geometry_y"]

        distances_km = np.sqrt(dx**2 + dy**2) / 1000

        candidates = candidates.copy()
        candidates["distance_km"] = distances_km

        candidates = candidates.sort_values("distance_km").head(k)

        # Distance-only summary
        distance_weights = np.exp(
            -candidates["distance_km"].to_numpy() / 25
        )

        weighted_damage = np.average(
            candidates["log_damage"],
            weights=distance_weights
        )

        # System-aware summaries
        same_river = candidates["river"] == row["river"]
        same_dijkring = candidates["dijkring"] == row["dijkring"]

        river_damage = (
            candidates.loc[same_river, "log_damage"].mean()
            if same_river.any()
            else candidates["log_damage"].mean()
        )

        dijkring_damage = (
            candidates.loc[same_dijkring, "log_damage"].mean()
            if same_dijkring.any()
            else candidates["log_damage"].mean()
        )

        output.append({
            "breach_id": row["breach_id"],
            "return_period": row["return_period"],
            "target_log_damage": row["log_damage"],

            "nearest_distance_km":
                candidates["distance_km"].min(),

            "mean_neighbor_distance_km":
                candidates["distance_km"].mean(),

            "distance_weighted_damage":
                weighted_damage,

            "same_river_neighbor_damage":
                river_damage,

            "same_dijkring_neighbor_damage":
                dijkring_damage,

            "mean_neighbor_damage":
                candidates["log_damage"].mean(),

            "n_neighbors":
                len(candidates)
        })

    return pd.DataFrame(output)


test_neighbor_features = build_neighbor_features(
    test,
    train,
    k=5
)

print(test_neighbor_features.head())
print("\nRows:", len(test_neighbor_features))

   breach_id  return_period  target_log_damage  nearest_distance_km  \
0         25            125           7.244942             1.621173   
1         25           1250           7.696667             1.621173   
2         25          12500           8.160804             1.621173   
3        212           1250           3.688879             3.074867   
4        220            125           9.472782             4.278523   

   mean_neighbor_distance_km  distance_weighted_damage  \
0                   6.017951                  7.839229   
1                   5.793395                  8.536045   
2                   6.715582                  8.644675   
3                  15.539741                  4.779152   
4                  12.340392                  9.236910   

   same_river_neighbor_damage  same_dijkring_neighbor_damage  \
0                    6.680019                       7.903729   
1                    7.613882                       8.538547   
2                    7.011296   

In [83]:
y_true = test_neighbor_features["target_log_damage"]

pred_distance = (
    test_neighbor_features["distance_weighted_damage"]
)

pred_river = (
    test_neighbor_features["same_river_neighbor_damage"]
)

pred_dijkring = (
    test_neighbor_features["same_dijkring_neighbor_damage"]
)

pred_combined = (
    0.5 * pred_river +
    0.5 * pred_dijkring
)

for name, pred in [
    ("Distance-weighted", pred_distance),
    ("Same-river", pred_river),
    ("Same-dijkring", pred_dijkring),
    ("Combined system", pred_combined)
]:

    mae = mean_absolute_error(y_true, pred)
    rmse = np.sqrt(mean_squared_error(y_true, pred))
    r2 = r2_score(y_true, pred)

    print(
        f"{name}: "
        f"MAE={mae:.4f}, "
        f"RMSE={rmse:.4f}, "
        f"R²={r2:.4f}"
    )

Distance-weighted: MAE=1.0790, RMSE=1.4266, R²=0.5940
Same-river: MAE=1.0809, RMSE=1.4453, R²=0.5833
Same-dijkring: MAE=1.1195, RMSE=1.5213, R²=0.5384
Combined system: MAE=1.0586, RMSE=1.4255, R²=0.5947


## What the first benchmark tells me

The combined system model is only marginally better than the distance-weighted model on this split (R² 0.5947 vs. 0.5940). I interpret this cautiously: much of the predictive signal is already captured by local spatial information, while river and dike-ring membership add only a small increment in this particular benchmark.

That does **not** mean system structure is irrelevant. The earlier dependence tests and the later event-generation experiments motivate keeping it in the representation even when its incremental predictive gain is small.

In [84]:
# Baseline 1: mean log damage for each return period,
# calculated using TRAIN data only.

rp_mean = (
    train
    .groupby("return_period")["log_damage"]
    .mean()
)

global_mean = train["log_damage"].mean()

pred_rp_mean = (
    test["return_period"]
    .map(rp_mean)
    .fillna(global_mean)
    .to_numpy()
)

y_true = test["log_damage"].to_numpy()

mae = mean_absolute_error(y_true, pred_rp_mean)
rmse = np.sqrt(mean_squared_error(y_true, pred_rp_mean))
r2 = r2_score(y_true, pred_rp_mean)

print("Return-period mean baseline:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

Return-period mean baseline:
MAE: 1.5434730091684752
RMSE: 1.8895664462506807
R²: 0.28777280175675524


In [85]:
pred_network = test_neighbor_features["distance_weighted_damage"].to_numpy()

mae_network = mean_absolute_error(y_true, pred_network)
rmse_network = np.sqrt(mean_squared_error(y_true, pred_network))
r2_network = r2_score(y_true, pred_network)

print("Network predictor:")
print("MAE:", mae_network)
print("RMSE:", rmse_network)
print("R²:", r2_network)

print("\nImprovement in MAE:")
print(
    (mae - mae_network) / mae * 100,
    "%"
)

print("\nImprovement in RMSE:")
print(
    (rmse - rmse_network) / rmse * 100,
    "%"
)

Network predictor:
MAE: 1.0789520119592573
RMSE: 1.4265993714970602
R²: 0.5940259500039697

Improvement in MAE:
30.095828981128232 %

Improvement in RMSE:
24.501232844827975 %


In [86]:
pred_global = np.full(
    len(test),
    train["log_damage"].mean()
)

print("Global mean:")
print(
    "MAE:",
    mean_absolute_error(y_true, pred_global)
)

print(
    "RMSE:",
    np.sqrt(mean_squared_error(y_true, pred_global))
)

print(
    "R²:",
    r2_score(y_true, pred_global)
)

Global mean:
MAE: 1.8815647672440163
RMSE: 2.2418075852700157
R²: -0.0025150153889375026


In [87]:
#repeated location-level cross-validation
from sklearn.model_selection import GroupShuffleSplit

def evaluate_split(train_df, test_df):
    features = build_neighbor_features(
        test_df,
        train_df,
        k=5
    )

    y_true = features["target_log_damage"].to_numpy()
    y_pred = features["distance_weighted_damage"].to_numpy()

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }


cv_results = []

gss = GroupShuffleSplit(
    n_splits=5,
    test_size=0.20,
    random_state=42
)

for split_id, (train_idx, test_idx) in enumerate(
    gss.split(
        model_data,
        groups=model_data["breach_id"]
    ),
    start=1
):

    train_cv = model_data.iloc[train_idx].copy()
    test_cv = model_data.iloc[test_idx].copy()

    metrics = evaluate_split(
        train_cv,
        test_cv
    )

    metrics["split"] = split_id
    metrics["train_locations"] = train_cv["breach_id"].nunique()
    metrics["test_locations"] = test_cv["breach_id"].nunique()

    cv_results.append(metrics)


cv_results = pd.DataFrame(cv_results)

print(cv_results)

        MAE      RMSE        R2  split  train_locations  test_locations
0  1.078952  1.426599  0.594026      1              475             119
1  1.333194  1.803519  0.407939      2              475             119
2  1.185466  1.543979  0.545647      3              475             119
3  1.296222  1.672666  0.474794      4              475             119
4  1.318918  1.801168  0.471963      5              475             119


In [88]:
print("\nMean performance:")
print(
    cv_results[
        ["MAE", "RMSE", "R2"]
    ].mean()
)

print("\nStd performance:")
print(
    cv_results[
        ["MAE", "RMSE", "R2"]
    ].std()
)


Mean performance:
MAE     1.242551
RMSE    1.649586
R2      0.498874
dtype: float64

Std performance:
MAE     0.108342
RMSE    0.164378
R2      0.072137
dtype: float64


## Location-level cross-validation

I repeat the location-level holdout procedure across five splits to check whether the apparent spatial signal is stable rather than a result of one favourable split.

In [89]:
baseline_cv_results = []

gss = GroupShuffleSplit(
    n_splits=5,
    test_size=0.20,
    random_state=42
)

for split_id, (train_idx, test_idx) in enumerate(
    gss.split(
        model_data,
        groups=model_data["breach_id"]
    ),
    start=1
):

    train_cv = model_data.iloc[train_idx].copy()
    test_cv = model_data.iloc[test_idx].copy()

    # Return-period mean learned ONLY from training locations
    rp_means = (
        train_cv
        .groupby("return_period")["log_damage"]
        .mean()
    )

    global_mean = train_cv["log_damage"].mean()

    y_true = test_cv["log_damage"].to_numpy()

    y_pred = (
        test_cv["return_period"]
        .map(rp_means)
        .fillna(global_mean)
        .to_numpy()
    )

    baseline_cv_results.append({
        "split": split_id,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(
            mean_squared_error(y_true, y_pred)
        ),
        "R2": r2_score(y_true, y_pred)
    })

baseline_cv_results = pd.DataFrame(
    baseline_cv_results
)

print(baseline_cv_results)

print("\nMean:")
print(
    baseline_cv_results[
        ["MAE", "RMSE", "R2"]
    ].mean()
)

print("\nStd:")
print(
    baseline_cv_results[
        ["MAE", "RMSE", "R2"]
    ].std()
)

   split       MAE      RMSE        R2
0      1  1.543473  1.889566  0.287773
1      2  1.738989  2.132159  0.172507
2      3  1.709297  2.071654  0.182016
3      4  1.700466  2.068489  0.196811
4      5  1.811765  2.216629  0.200272

Mean:
MAE     1.700798
RMSE    2.075700
R2      0.207876
dtype: float64

Std:
MAE     0.098229
RMSE    0.120169
R2      0.046053
dtype: float64


In [90]:
comparison = cv_results.merge(
    baseline_cv_results,
    on="split",
    suffixes=("_spatial", "_rp")
)

comparison["MAE_improvement_pct"] = (
    (comparison["MAE_rp"] - comparison["MAE_spatial"])
    / comparison["MAE_rp"]
    * 100
)

comparison["RMSE_improvement_pct"] = (
    (comparison["RMSE_rp"] - comparison["RMSE_spatial"])
    / comparison["RMSE_rp"]
    * 100
)

comparison["R2_gain"] = (
    comparison["R2_spatial"]
    - comparison["R2_rp"]
)

print(comparison)

   MAE_spatial  RMSE_spatial  R2_spatial  split  train_locations  \
0     1.078952      1.426599    0.594026      1              475   
1     1.333194      1.803519    0.407939      2              475   
2     1.185466      1.543979    0.545647      3              475   
3     1.296222      1.672666    0.474794      4              475   
4     1.318918      1.801168    0.471963      5              475   

   test_locations    MAE_rp   RMSE_rp     R2_rp  MAE_improvement_pct  \
0             119  1.543473  1.889566  0.287773            30.095829   
1             119  1.738989  2.132159  0.172507            23.335094   
2             119  1.709297  2.071654  0.182016            30.645989   
3             119  1.700466  2.068489  0.196811            23.772509   
4             119  1.811765  2.216629  0.200272            27.202586   

   RMSE_improvement_pct   R2_gain  
0             24.501233  0.306253  
1             15.413488  0.235432  
2             25.471197  0.363631  
3             

## Interpretation

Across five breach-location holdout splits, the spatial-neighbour baseline achieves approximately R² = 0.50 ± 0.07. Compared with the return-period-only baseline, the spatial model reduces MAE by about 27% and RMSE by about 21%.

For me, the important result is not that the spatial baseline is already a finished model. It is that **location information provides repeatable out-of-sample signal beyond return period**, which supports moving to a richer spatial/system representation for the next stage of the project.

## Takeaway and transition to modelling

This notebook established the part of the problem that I needed before introducing a more complex model: flood-consequence similarity is spatially structured, system membership matters, and spatial information remains useful when I hold out entire breach locations.

The next question is therefore no longer **whether** there is structure, but **how to represent and learn it well enough to generate dependent multi-breach events**. That motivates the Random Forest, MLP and graph-based experiments in the next stage.